# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [12]:
# Write your code below.
%load_ext dotenv
%dotenv


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [13]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [14]:
import os
from glob import glob

# Write your code below.

PRICE_DATA = os.getenv("PRICE_DATA")
PRICE_DATA
if not PRICE_DATA:
    raise ValueError("PRICE_DATA environment variable is not set.")
parquet_files = glob (
    os.path.join(PRICE_DATA,"**","*.parquet"),
    recursive=True
)

print(f"Found {len(parquet_files)} parquet files in {PRICE_DATA}")

Found 2871 parquet files in ../../05_src/data/prices/


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [30]:
# Write your code below.
#loading the files and prints the columns of the dataframe
dd_price = dd.read_parquet(parquet_files)
print (dd_price.columns.tolist())
#we need to rename the column "Adj Close" to "Adj_Close" because it has a space in it which can cause issues when working with the dataframe. 
#The code below checks if the column exists and renames it if it does.
if "Adj Close" in dd_price.columns:
    dd_price = dd_price.rename(columns={"Adj Close": "Adj_Close"})

#Group using ticker as the index
dd_price = dd_price.set_index("ticker")
#Add lags for variables Close and Adj_Close.
#Add returns and hi_lo_range features to the dataframe.
import pandas as pd
dd_lags = (
    dd_price
    .groupby("ticker", group_keys=False)
    .apply(
        lambda x: x.sort_values("Date").assign(
            Close_lag_1=lambda df: df["Close"].shift(1),
            Adj_Close_lag_1=lambda df: df["Adj_Close"].shift(1)
        ),
        meta=dd_price._meta.assign(
            Close_lag_1=pd.Series(dtype="float64"),
            Adj_Close_lag_1=pd.Series(dtype="float64")
        )
    )
)
#Assign the result to `dd_feat` 
dd_feat = dd_lags.assign(
    returns=lambda df: df["Close"] / df["Close_lag_1"] - 1,
    hi_lo_range=lambda df: df["High"] - df["Low"]
)

print (dd_lags.columns.tolist())
print (dd_feat.columns.tolist())

['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source', 'ticker', 'Year']
['Date', 'Open', 'High', 'Low', 'Close', 'Adj_Close', 'Volume', 'source', 'Year', 'Close_lag_1', 'Adj_Close_lag_1']
['Date', 'Open', 'High', 'Low', 'Close', 'Adj_Close', 'Volume', 'source', 'Year', 'Close_lag_1', 'Adj_Close_lag_1', 'returns', 'hi_lo_range']


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [31]:
# Write your code below.
#convert Dask to pandas

df = dd_feat.compute()


In [32]:
df.head ()

,Date,Open,High,Low,Close,Adj_Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
ticker,,,,,,,,,,,,,
ACEL,2019-03-11,10.15,10.15,10.150,10.15,10.15,46300.0,ACEL.csv,2019,NaN,NaN,NaN,0.000
ACEL,2019-03-12,10.13,10.15,10.080,10.15,10.15,12300.0,ACEL.csv,2019,10.15,10.15,0.000000,0.070
ACEL,2019-03-13,10.15,10.15,10.148,10.15,10.15,453500.0,ACEL.csv,2019,10.15,10.15,0.000000,0.002
ACEL,2019-03-14,10.14,10.18,10.140,10.14,10.14,2100.0,ACEL.csv,2019,10.15,10.15,-0.000985,0.040
ACEL,2019-03-15,10.16,10.16,10.160,10.16,10.16,100.0,ACEL.csv,2019,10.14,10.14,0.001972,0.000


In [35]:
df = (
    df.groupby("ticker", group_keys=False)
    .apply(
        lambda x: x.sort_values("Date").assign(
            returns_ma_10=lambda df: df["returns"].rolling(10).mean()
        )
    )
)
#df.head ()
df[["Date", "returns", "returns_ma_10"]].head(15)

,Date,returns,returns_ma_10
ticker,,,
ACEL,2019-03-11,NaN,NaN
ACEL,2019-03-12,0.000000,NaN
ACEL,2019-03-13,0.000000,NaN
ACEL,2019-03-14,-0.000985,NaN
ACEL,2019-03-15,0.001972,NaN
ACEL,2019-03-18,0.001969,NaN
ACEL,2019-03-19,0.000000,NaN
ACEL,2019-03-20,0.000000,NaN
ACEL,2019-03-21,-0.003929,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

My answer: 

It was not necessary to convert to pandas because Dask can also calculate a moving average. In this case, pandas was easier to use because the data fit in memory. For a much larger dataset, Dask would be a better choice because it can process data in smaller parts. However, we would need to make sure the returns are ordered by date and calculated separately for each ticker, including across those parts.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.